In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# 26_580_643 rows

In [3]:

iid = 0
SAVE_OUT_PATH = "/scratch/midway3/rmastand/muon_collider/nuGun_pT_0_50/reco_h5/"
path_to_data = f"{SAVE_OUT_PATH}/nuGun_pT_0_50_reco_{iid}.h5"
df_head = pd.read_hdf(path_to_data, key="df", stop=10)
print(df_head)


   event                    collection      Edep          t           x  \
0      0  InnerTrackerBarrelCollection  0.000028   4.001957 -503.471520   
1      0  InnerTrackerBarrelCollection  0.000035   3.993593 -503.892225   
2      0  InnerTrackerBarrelCollection  0.000125   2.825821 -549.297765   
3      0  InnerTrackerBarrelCollection  0.000079   2.749115  159.376294   
4      0  InnerTrackerBarrelCollection  0.000204   2.748186  159.384792   
5      0  InnerTrackerBarrelCollection  0.000036   2.741781  160.211095   
6      0  InnerTrackerBarrelCollection  0.000101   0.799673   61.521157   
7      0  InnerTrackerBarrelCollection  0.000059  14.911034 -127.009552   
8      0  InnerTrackerBarrelCollection  0.000062   2.214822  243.244621   
9      0  InnerTrackerBarrelCollection  0.000026   0.857233  116.482556   

            y           z  system  side  layer  module  sensor    cellid0  \
0 -231.515346 -617.124690       3     0      2      70       2   34128131   
1 -230.534940 -615.5

In [ ]:
collections = [
     "OuterTrackerBarrelCollection",     
    "OuterTrackerEndcapCollection",  
    "InnerTrackerBarrelCollection",
    "InnerTrackerEndcapCollection",
    "VertexBarrelCollection",   
    "VertexEndcapCollection",
]

OUTPUT_COLS = ["Edep", "x", "y", "z", "t", "system", "side", "layer", "module", "sensor"]

results = {col: [] for col in collections}
counts_all = {col: 0 for col in collections}

for chunk in pd.read_hdf(path_to_data, key="df", chunksize=1_000_000):
    for col in collections:
        mask_all = chunk["collection"] == col
        counts_all[col] += mask_all.sum()
        mask = mask_all & (chunk["inside_bounds"] == True)
        results[col].append(chunk.loc[mask, OUTPUT_COLS])

for col in collections:
    print(f"length of {col}, all: {counts_all[col]}")
    arr = pd.concat(results[col]).to_numpy() if results[col] else np.empty((0, len(OUTPUT_COLS)))
    print(f"length of {col}, inside_bounds: {len(arr)}")
    np.save(f"{SAVE_OUT_PATH}/{col}_SimTrackerHit_conditional_reco_{iid}.npy", arr)